# Pokémon Rojo Fuego Full + B10 — ROM privada para MyBoy

**Personal, experimental; no es una versión final.** Esta versión realiza **una comprobación previa de Full sin importar gráficos**. Solo si Full coincide con el CI, descarga las 386 parejas normales/shiny, compila el juego integrado y verifica el SHA-1 de la ROM B10. Ninguna ROM ni sprite se sube a GitHub.

**Pasos en Android:** pulsa ▶ en la primera celda de código. Si aparece `ROM_VERIFICADA_CONTRA_CI`, la última celda descarga el archivo `.gba`. Si aparece **cualquier error**, la última celda descarga **solo un informe `.json`** para diagnosticarlo, nunca una ROM no verificada. En caso de fallo no repitas el bloque entero sin revisar el informe. Los archivos temporales de Colab se pierden cuando se reinicia su entorno.

No necesitas conectar Google Drive, pegar contraseñas ni modificar archivos. La comprobación no concede derechos de redistribución del juego o sus gráficos.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess

REPO = "https://github.com/joseiturriaga25amz/pokefirered-europe.git"
BRANCH = "diagnostic/b10-full-integrated-candidate"
PIN = "e94824cf7afcf8062c383f6af72dc5c7cf37852f"
CI_SHA1 = "6fed3237225dc6146649d85c969b1e9da4352126"
OUTPUT = Path("/content/b10-myboy-privado/Pokemon_Rojo_Fuego_Full_B10_386.gba")
SOURCE = Path("/content/b10-full-source")

def cmd(*args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def read(*args, cwd=None):
    return subprocess.check_output(args, cwd=cwd, text=True).strip()

if SOURCE.exists() or OUTPUT.exists() or OUTPUT.with_suffix(".diagnostic.json").exists():
    raise RuntimeError("Ya hay una carpeta/ROM de una ejecucion anterior. Reinicia el entorno de Colab para evitar mezclar compilaciones.")

cmd("apt-get", "-o", "Acquire::Retries=3", "update")
cmd("apt-get", "-o", "Acquire::Retries=3", "install", "-y", "git", "make", "gcc", "pkg-config", "gcc-arm-none-eabi", "binutils-arm-none-eabi", "libpng-dev")
cmd("git", "clone", "--depth", "50", "--single-branch", "--branch", BRANCH, REPO, str(SOURCE))
if read("git", "branch", "--show-current", cwd=SOURCE) != BRANCH:
    raise RuntimeError("Rama inesperada")
cmd("git", "cat-file", "-e", PIN + "^{commit}", cwd=SOURCE)
cmd("git", "reset", "--hard", PIN, cwd=SOURCE)
if read("git", "rev-parse", "HEAD", cwd=SOURCE) != PIN:
    raise RuntimeError("Commit incorrecto")
cmd("python3", "tools/build_follower_diagnostic.py", "--self-test", cwd=SOURCE)

# Verificacion acotada: la ROM Full, sin sprites adicionales, debe
# reproducir el CI antes de descargar las 386 parejas de graficos.
import os, shutil, tempfile, sys

BASELINE_CI_SHA1 = "511f902a9f14f318e62623caa54d82af89ec48d0"
REPORT = Path("/content/b10-myboy-privado/diagnostico_compilacion.json")
REPORT.parent.mkdir(parents=True, exist_ok=True)

def program_version(name):
    executable = shutil.which(name)
    if executable is None:
        return {"path": None, "version": "NO DISPONIBLE"}
    result = subprocess.run([executable, "--version"], text=True,
                            capture_output=True, check=False)
    lines = (result.stdout or result.stderr).splitlines()
    return {"path": executable, "version": lines[0] if lines else "SIN VERSION"}

diagnostic = {
    "schema": "B10_PRIVATE_COLAB_DIAGNOSTIC_V1",
    "status": "PREFLIGHT_STARTED",
    "project_git_sha": PIN,
    "full_ci_sha1_expected": BASELINE_CI_SHA1,
    "b10_ci_sha1_expected": CI_SHA1,
    "python": sys.version.split()[0],
    "os_release": Path("/etc/os-release").read_text().splitlines()[:10],
    "toolchain": {
        key: program_version(key)
        for key in ("arm-none-eabi-gcc", "arm-none-eabi-as",
                    "arm-none-eabi-ld", "gcc", "make", "pkg-config")
    },
    "packages": subprocess.run(
        ["dpkg-query", "-W", "gcc-arm-none-eabi",
         "binutils-arm-none-eabi", "libpng-dev"],
        capture_output=True, text=True, check=False
    ).stdout.strip().splitlines(),
    "locale": {key: os.getenv(key) for key in ("LANG", "LC_ALL")},
}
def record(status, **values):
    diagnostic.update(values)
    diagnostic["status"] = status
    REPORT.write_text(json.dumps(diagnostic, indent=2, ensure_ascii=False,
                                 sort_keys=True) + "\n")
    print("DIAGNOSTICO:", status)
    print("Informe privado:", REPORT)

record("PREFLIGHT_STARTED")
for tool in ("arm-none-eabi-gcc", "arm-none-eabi-as", "arm-none-eabi-ld"):
    if not diagnostic["toolchain"][tool]["path"]:
        record("PREFLIGHT_BLOCKED_MISSING_ARM", missing=tool)
        raise RuntimeError("Faltan herramientas ARM; revisar informe. NO descargar ROM")

def compile_plain_full(checkout):
    result = subprocess.run(
        ["make", "-j2", "firered_es_modern"], cwd=checkout,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        check=False
    )
    rom = checkout / "pokefirered_modern_es.gba"
    if result.returncode != 0 or not rom.is_file():
        record("BASELINE_BUILD_FAILED",
               build_returncode=result.returncode,
               last_build_lines=result.stdout.splitlines()[-20:])
        raise RuntimeError("Fallo Full base; NO descargar ROM. Revisar informe")
    return hashlib.sha1(rom.read_bytes()).hexdigest()

with tempfile.TemporaryDirectory(prefix="b10-full-preflight-", dir="/content") as temp:
    checkout = Path(temp) / "checkout"
    added = False
    try:
        cmd("git", "worktree", "add", "--detach", str(checkout), PIN, cwd=SOURCE)
        added = True
        if read("git", "rev-parse", "HEAD", cwd=checkout) != PIN:
            record("BASELINE_WRONG_COMMIT")
            raise RuntimeError("Commit de Full inesperado")
        first_sha = compile_plain_full(checkout)
        record("BASELINE_FIRST_BUILD_COMPLETE", full_first_sha1=first_sha)
        if first_sha != BASELINE_CI_SHA1:
            clean = subprocess.run(["make", "clean"], cwd=checkout,
                                   capture_output=True, text=True, check=False)
            if clean.returncode != 0:
                record("BASELINE_CLEAN_FAILED",
                       last_clean_lines=(clean.stdout + clean.stderr).splitlines()[-20:])
                raise RuntimeError("No se pudo limpiar la primera compilacion")
            second_sha = compile_plain_full(checkout)
            record("BLOCKED_BASELINE_SHA_MISMATCH",
                   full_second_sha1=second_sha,
                   full_local_reproducible=(first_sha == second_sha))
            print("FULL_CI_ESPERADO:", BASELINE_CI_SHA1)
            print("FULL_LOCAL_1:", first_sha)
            print("FULL_LOCAL_2:", second_sha)
            raise RuntimeError("Full base no coincide con CI: se omite la descarga de graficos B10. Ejecuta solo la celda de informe")
        record("BASELINE_CI_VERIFIED", full_first_sha1=first_sha)
        print("FULL_BASE_VERIFICADO_CONTRA_CI", first_sha)
    finally:
        if added:
            cmd("git", "worktree", "remove", "--force", str(checkout), cwd=SOURCE)

# Solo ahora se permite descargar la fuente fijada de 386 normal/shiny.
try:
    cmd("python3", "tools/build_follower_diagnostic.py", "--download-pinned", "--output", str(OUTPUT), cwd=SOURCE)
except Exception as exc:
    record("B10_BUILD_FAILED", failure_type=type(exc).__name__, failure_message=str(exc))
    raise
manifest = json.loads(OUTPUT.with_suffix(".diagnostic.json").read_text())
rom = OUTPUT.read_bytes()
if not (0 < len(rom) <= 32 * 1024 * 1024):
    raise RuntimeError("Tamano de ROM fuera de rango")
if manifest.get("project_git_sha") != PIN or manifest.get("national_species_count") != 386 or manifest.get("normal_shiny_variants") != 2:
    raise RuntimeError("Identidad del proyecto o cobertura incorrecta")
if manifest.get("source_feasible_species") != 386 or manifest.get("source_blocked_species") != 0:
    raise RuntimeError("Cobertura grafica 386/386 incompleta")
if manifest.get("rom_bytes") != len(rom) or manifest.get("rom_sha1") != hashlib.sha1(rom).hexdigest() or manifest.get("rom_sha256") != hashlib.sha256(rom).hexdigest():
    raise RuntimeError("La huella de la ROM no coincide con su manifiesto")
if manifest["rom_sha1"] != CI_SHA1:
    record("BLOCKED_B10_SHA_MISMATCH",
           b10_actual_sha1=manifest["rom_sha1"],
           b10_actual_sha256=manifest["rom_sha256"],
           b10_bytes=manifest["rom_bytes"],
           source_archive_sha256=manifest["source_archive_sha256"])
    OUTPUT.unlink()
    raise RuntimeError("B10 difiere del CI: ROM bloqueada y descartada. Ejecuta solo la celda de informe")
if (SOURCE / "pokefirered_modern_es.gba").exists() or (SOURCE / "graphics/object_events/pics/follower_import").exists():
    raise RuntimeError("Contaminacion del directorio fuente")
record("B10_CI_VERIFIED", b10_actual_sha1=manifest["rom_sha1"],
       b10_actual_sha256=manifest["rom_sha256"], b10_bytes=len(rom),
       source_archive_sha256=manifest["source_archive_sha256"])
print("ROM_VERIFICADA_CONTRA_CI", "SHA-1:", manifest["rom_sha1"], "Bytes:", len(rom))
print("Archivo privado listo para descarga:", OUTPUT)
print("No hay prueba de MyBoy ni autorizacion de redistribucion de sprites.")


## Descargar ROM verificada **o** informe de diagnóstico

Ejecuta esta última celda después de la anterior: **si la ROM coincide exactamente con el CI**, se descargará un único `.gba` para abrir en MyBoy. **Si falla la compilación o la huella**, descargará **solo `diagnostico_compilacion.json`**, sin sprites ni ROM; compártenos ese informe para investigar. Si Colab se reinicia, los archivos temporales desaparecen. MyBoy todavía debe probarse por separado.

In [ ]:
from pathlib import Path
import hashlib, json
from google.colab import files

report_path = Path("/content/b10-myboy-privado/diagnostico_compilacion.json")
rom_path = Path("/content/b10-myboy-privado/Pokemon_Rojo_Fuego_Full_B10_386.gba")
expected = "6fed3237225dc6146649d85c969b1e9da4352126"

if not report_path.is_file():
    raise RuntimeError("No hay informe: ejecuta primero la celda de compilacion en esta sesion")
result = json.loads(report_path.read_text())
if result.get("status") != "B10_CI_VERIFIED":
    print("SOLO_INFORME_DE_DIAGNOSTICO; NO se descarga ninguna ROM")
    files.download(str(report_path))
else:
    if (not rom_path.is_file()
            or result.get("b10_actual_sha1") != expected
            or hashlib.sha1(rom_path.read_bytes()).hexdigest() != expected):
        raise RuntimeError("La ROM ha desaparecido o no coincide con CI: descarga bloqueada")
    print("DESCARGA_ROM_PRIVADA_VERIFICADA", expected)
    files.download(str(rom_path))
